# EPPS — DRF + DEG-Jaccard figure reproduction (end-to-end)

The datasets are downloaded **already preprocessed** from a public GCS bucket
(`gs://perturb-seq-datasets/epps/processed`) — log-normalized `X`, a `perturbation` obs
column, and a `control` label; see the bucket's `README.md` for the format. Preprocessing
is out of scope for this notebook.

Pipeline: (1) install EPPS, (2) download the datasets, (3) run DRF for the 7 datasets +
DE export for the 4 Jaccard datasets, (4) render and display the figures. Deterministic
(seed 42, 8192 subsample).

## Setup — install EPPS

In [ ]:
import subprocess, sys
from pathlib import Path

HERE = Path.cwd()                      # run this notebook from EPPS/publication/
EPPS_DIR = HERE.parent                 # the EPPS package repo root (this notebook lives in EPPS/publication/)
assert (EPPS_DIR / 'pyproject.toml').exists(), f'expected the EPPS repo root at {EPPS_DIR}'

# Standard install (pulls anndata/scanpy/torch/geomloss/illico/... from pyproject).
subprocess.run([sys.executable, '-m', 'pip', 'install', '-e', str(EPPS_DIR), 'matplotlib'], check=True)
PY = sys.executable                    # everything runs through this kernel's Python

## Configuration

In [ ]:
FIG_ROOT = HERE
DATA_DIR = FIG_ROOT / 'data'           # downloaded datasets land here
DRF_OUT  = FIG_ROOT / 'drf_outputs'
DE_OUT   = FIG_ROOT / 'de_outputs'
for d in (DATA_DIR, DRF_OUT, DE_OUT): d.mkdir(exist_ok=True)

BUCKET = 'gs://perturb-seq-datasets/epps/processed'
def dataset(ds): return DATA_DIR / f'{ds}_processed_complete.h5ad'

def run(cmd, cwd=None):
    print('$', ' '.join(str(c) for c in cmd))
    subprocess.run([str(c) for c in cmd], cwd=cwd, check=True)

## Step 1 — download the datasets

Pull the standardized (already-preprocessed) `.h5ad` files from the bucket into `DATA_DIR`.
Requires the gcloud SDK / `gsutil` on `PATH`.

In [ ]:
run(['gsutil', '-m', 'cp', f'{BUCKET}/*_processed_complete.h5ad', f'{DATA_DIR}/'])

## Step 2 — EPPS DRF for the 7 datasets

`epps run -p all --profile` computes all 22 protocols + a per-protocol timing table per
dataset. **Long-running** (Sinkhorn dominates).

In [ ]:
DRF_DATASETS = ['wessels23', 'replogle22rpe1', 'replogle22k562',
                'nadig25jurkat', 'nadig25hepg2', 'arch1', 'kaden25rpe1']
for ds in DRF_DATASETS:
    print(f'=== DRF {ds} ===')
    run([PY, '-m', 'epps', 'run', dataset(ds), '-p', 'all', '--profile',
         '--subsample', 8192, '--seed', 42, '--output', 'drf', '--out-dir', DRF_OUT, '--quiet'])

## Step 3 — EPPS DE export for the 4 DEG-Jaccard datasets

`epps de` recomputes per-gene DE (t-test + MWU, GT first-half vs the 8192 pool) to HDF5.

In [ ]:
DE_DATASETS = ['wessels23', 'replogle22k562', 'nadig25hepg2', 'arch1']
for ds in DE_DATASETS:
    print(f'=== DE {ds} ===')
    run([PY, '-m', 'epps', 'de', dataset(ds), '--methods', 't-test,MWU',
         '--subsample', 8192, '--seed', 42, '--out-dir', DE_OUT])

## Step 4 — render the figures

In [ ]:
for script in ['drf_table_figure.py', 'timing_table.py', 'deg_jaccard_figure.py']:
    run([PY, FIG_ROOT / script], cwd=FIG_ROOT)

## Step 5 — display the figures

In [ ]:
from IPython.display import Image, display
for fig in ['drf_table_mean.png', 'drf_table_median.png',
            'metric_timing_table.png', 'deg_jaccard_with_counts.png']:
    print(fig)
    display(Image(filename=str(FIG_ROOT / 'figures' / fig)))